In [1]:
import torch

# Kiểm tra xem CUDA (GPU) có khả dụng không
cuda_available = torch.cuda.is_available()
print(f"CUDA khả dụng: {cuda_available}")

# Nếu có GPU, in thêm thông tin chi tiết
if cuda_available:
    print(f"-> Tên GPU: {torch.cuda.get_device_name(0)}")
    # In phiên bản CUDA mà PyTorch được biên dịch cùng
    # Lưu ý: Phiên bản này có thể khác một chút so với CUDA Toolkit bạn cài đặt
    print(f"-> Phiên bản CUDA (PyTorch): {torch.version.cuda}")
else:
    print("-> ⚠️ Không tìm thấy GPU tương thích CUDA. PyTorch sẽ sử dụng CPU.")

CUDA khả dụng: True
-> Tên GPU: NVIDIA GeForce RTX 4050 Laptop GPU
-> Phiên bản CUDA (PyTorch): 13.0


In [ ]:
from flask import Flask, request, jsonify
import cv2
import numpy as np
from ultralytics import YOLO
import joblib
import threading
import time
import os
from queue import Queue, Full, Empty # Đảm bảo import Empty

# --- 1. CẤU HÌNH ---
YOLO_MODEL_PATH = 'yolo11n-pose.pt'
CLASSIFIER_PATH = 'posture_classifier.joblib'
SCALER_PATH = 'posture_scaler.joblib'

# Kích thước ảnh đầu vào (PHẢI GIỐNG LÚC TRAIN & ESP32)
INPUT_WIDTH = 320
INPUT_HEIGHT = 240
TARGET_SIZE = (INPUT_WIDTH, INPUT_HEIGHT)

# Keypoints quan trọng (PHẢI GIỐNG LÚC TRAIN)
IMPORTANT_KEYPOINTS_INDICES = [0, 5, 6, 11, 12] # Mũi, Vai trái/phải, Hông trái/phải
# Chiều dài vector đặc trưng mong đợi sau khi chuẩn hóa và bỏ mũi
EXPECTED_FEATURE_LENGTH = (len(IMPORTANT_KEYPOINTS_INDICES) - (1 if IMPORTANT_KEYPOINTS_INDICES[0] == 0 else 0) ) * 2

# --- CẤU HÌNH THÔNG BÁO & LƯU TRỮ ---
INCORRECT_POSTURE_THRESHOLD_SECONDS = 60 # Chỉ thông báo sau 60 giây ngồi sai liên tục
CORRECT_POSTURE_RESET_ALERT_SECONDS = 30 # Reset cảnh báo sau 30 giây ngồi đúng liên tục
SAVE_IMAGE_ON_ALERT = True # Có lưu ảnh khi cảnh báo không?
SAVE_PATH_CORRECT = "saved_images/correct/"
SAVE_PATH_INCORRECT = "saved_images/incorrect/"
SHOW_PREVIEW_WINDOW = True # Đặt là True để xem ảnh xử lý trên màn hình PC

# Tạo thư mục lưu ảnh
os.makedirs(SAVE_PATH_CORRECT, exist_ok=True)
os.makedirs(SAVE_PATH_INCORRECT, exist_ok=True)

# --- 2. TẢI MODEL ---
print("Đang tải mô hình...")
try:
    # Ultralytics sẽ tự động dùng GPU nếu PyTorch được cài đặt đúng
    yolo_model = YOLO(YOLO_MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    classifier = joblib.load(CLASSIFIER_PATH)
    print("-> Mô hình YOLO, Scaler, và Classifier đã tải thành công.")
    # Kiểm tra nhanh GPU
    import torch
    if torch.cuda.is_available():
        print(f"   -> Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("   -> ⚠️ Cảnh báo: Không tìm thấy GPU, đang sử dụng CPU.")
except Exception as e:
    print(f"❌ Lỗi tải model: {e}")
    exit()

# --- 3. KHỞI TẠO FLASK VÀ CÁC BIẾN TOÀN CỤC ---
app = Flask(__name__)
frame_queue = Queue(maxsize=1) # Queue chỉ giữ frame mới nhất

# Biến theo dõi trạng thái
latest_frame_processed = None
posture_status = 2 # 0: OK, 1: Sai, 2: Không phát hiện/Lỗi
incorrect_posture_start_time = None
alert_sent_for_current_session = False
correct_posture_start_time = None
last_saved_correct_image_time = 0

# --- 4. HÀM TRÍCH XUẤT ĐẶC TRƯNG TỪ FRAME ---
def extract_pose_features_from_frame(frame, yolo_model):
    """Trích xuất và chuẩn hóa keypoints từ frame ảnh."""
    try:
        if frame is None: return None
        # Giả định frame từ ESP32 đã đúng size (320x240)
        img_resized = frame

        results = yolo_model(img_resized, verbose=False)

        if results and results[0].keypoints and results[0].keypoints.shape[1] > 0:
            keypoints = results[0].keypoints.xy[0].cpu().numpy()
            features = []
            for idx in IMPORTANT_KEYPOINTS_INDICES:
                if idx < len(keypoints) and keypoints[idx][0] > 1 and keypoints[idx][1] > 1:
                    features.extend(keypoints[idx])
                else:
                    features.extend([0, 0])

            # Chuẩn hóa theo mũi
            if len(features) > 1 and IMPORTANT_KEYPOINTS_INDICES[0] == 0 and features[0] != 0 and features[1] != 0:
                 nose_x, nose_y = features[0], features[1]
                 normalized_features = []
                 for i in range(0, len(features), 2):
                      normalized_features.append(features[i] - nose_x if features[i] != 0 else 0)
                      normalized_features.append(features[i+1] - nose_y if features[i+1] != 0 else 0)
                 final_features = np.array(normalized_features[2:])
                 return final_features if len(final_features) == EXPECTED_FEATURE_LENGTH else None
            elif len(features) > 0:
                 final_features = np.array(features)
                 return final_features if len(final_features) == EXPECTED_FEATURE_LENGTH else None
            else: return None
        else: return None
    except Exception as e:
        print(f"Lỗi trích xuất đặc trưng: {e}")
        return None

# --- 5. HÀM DỰ ĐOÁN TƯ THẾ ---
def predict_posture(frame, yolo_model, scaler, classifier):
    """Trích xuất features, chuẩn hóa và dự đoán tư thế."""
    start_time = time.time() # Bắt đầu đo thời gian xử lý
    features = extract_pose_features_from_frame(frame, yolo_model)
    feature_time = time.time()

    status_code = 2 # Mặc định là không phát hiện
    if features is not None:
        try:
            features_reshaped = features.reshape(1, -1)
            features_scaled = scaler.transform(features_reshaped)
            prediction = classifier.predict(features_scaled)[0]
            status_code = int(prediction) # 0 hoặc 1
        except Exception as e:
            print(f"Lỗi dự đoán SVM: {e}")
            status_code = 2 # Lỗi
    
    end_time = time.time()
    yolo_duration = (feature_time - start_time) * 1000
    svm_duration = (end_time - feature_time) * 1000
    total_duration = (end_time - start_time) * 1000
    # print(f"⏱️ Xử lý: YOLO {yolo_duration:.1f}ms + SVM {svm_duration:.1f}ms = {total_duration:.1f}ms") # Bỏ comment để xem tốc độ
    return status_code

# --- 6. LUỒNG XỬ LÝ ẢNH NỀN ---
def background_processor():
    global latest_frame_processed, posture_status
    global incorrect_posture_start_time, alert_sent_for_current_session
    global correct_posture_start_time, last_saved_correct_image_time

    while True:
        try:
            frame_to_process = frame_queue.get(block=True)
            current_status = predict_posture(frame_to_process, yolo_model, scaler, classifier)
            posture_status = current_status
            latest_frame_processed = frame_to_process.copy()

            # --- Logic Cảnh báo theo thời gian ---
            current_time = time.time()
            if current_status == 1: # Sai
                correct_posture_start_time = None
                if incorrect_posture_start_time is None: incorrect_posture_start_time = current_time
                else:
                    duration_incorrect = current_time - incorrect_posture_start_time
                    if duration_incorrect >= INCORRECT_POSTURE_THRESHOLD_SECONDS and not alert_sent_for_current_session:
                        print(f"[{time.strftime('%H:%M:%S')}] !!! GỬI CẢNH BÁO TƯ THẾ SAI !!!")
                        alert_sent_for_current_session = True
                        if SAVE_IMAGE_ON_ALERT:
                            timestamp = time.strftime("%Y%m%d_%H%M%S")
                            filename = os.path.join(SAVE_PATH_INCORRECT, f"incorrect_{timestamp}.jpg")
                            try: cv2.imwrite(filename, frame_to_process); print(f"-> Đã lưu ảnh sai: {filename}")
                            except Exception as e_save: print(f"Lỗi lưu ảnh sai: {e_save}")

            elif current_status == 0: # Đúng
                incorrect_posture_start_time = None
                if correct_posture_start_time is None: correct_posture_start_time = current_time
                else:
                    duration_correct = current_time - correct_posture_start_time
                    if alert_sent_for_current_session and duration_correct >= CORRECT_POSTURE_RESET_ALERT_SECONDS:
                        print(f"[{time.strftime('%H:%M:%S')}] Ngồi đúng đủ lâu, reset cờ cảnh báo.")
                        alert_sent_for_current_session = False
                if SAVE_IMAGE_ON_ALERT and (current_time - last_saved_correct_image_time > 300): # Lưu ảnh đúng mỗi 5 phút
                     timestamp = time.strftime("%Y%m%d_%H%M%S")
                     filename = os.path.join(SAVE_PATH_CORRECT, f"correct_{timestamp}.jpg")
                     try: cv2.imwrite(filename, frame_to_process); print(f"-> Đã lưu ảnh đúng: {filename}"); last_saved_correct_image_time = current_time
                     except Exception as e_save: print(f"Lỗi lưu ảnh đúng: {e_save}")

            else: # Không phát hiện
                incorrect_posture_start_time = None
                correct_posture_start_time = None
                if alert_sent_for_current_session:
                     print(f"[{time.strftime('%H:%M:%S')}] Không thấy người, reset cờ cảnh báo.")
                     alert_sent_for_current_session = False

            frame_queue.task_done()
        except Exception as e:
            print(f"Lỗi trong luồng xử lý nền: {e}")
            posture_status = 2
            time.sleep(1)

# --- 7. ROUTE NHẬN ẢNH TỪ ESP32/CLIENT ---
@app.route('/', methods=['POST'])
def receive_image():
    try:
        img_bytes = request.data
        if not img_bytes:
            return jsonify({"status": "error", "message": "No image data received"}), 400

        nparr = np.frombuffer(img_bytes, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if frame is not None:
            # Kiểm tra kích thước ảnh nhận được (để debug)
            # print(f"Received frame size: {frame.shape[1]}x{frame.shape[0]}")
            if frame.shape[1] != INPUT_WIDTH or frame.shape[0] != INPUT_HEIGHT:
                 print(f"⚠️ Cảnh báo: Kích thước frame nhận được ({frame.shape[1]}x{frame.shape[0]}) khác TARGET_SIZE ({INPUT_WIDTH}x{INPUT_HEIGHT}). Đang resize...")
                 frame = cv2.resize(frame, TARGET_SIZE) # Resize nếu cần

            try:
                # Cố gắng lấy frame cũ ra trước khi đặt frame mới vào
                try: frame_queue.get_nowait()
                except Empty: pass
                frame_queue.put_nowait(frame)
            except Full: # Rất hiếm khi xảy ra với queue size 1 và get_nowait
                pass 
                
            # Trả về trạng thái cảnh báo cuối cùng
            status_to_send_esp = 1 if alert_sent_for_current_session else posture_status
            message = "GOOD_POSTURE" if status_to_send_esp == 0 else "BAD_POSTURE_ALERT" if status_to_send_esp == 1 else "NO_PERSON/ERROR"

            return jsonify({"status_code": status_to_send_esp, "message": message}), 200
        else:
            print("Lỗi giải mã JPEG nhận được.")
            return jsonify({"status": "error", "message": "Lỗi giải mã JPEG"}), 400
    except Exception as e:
        print(f"Lỗi server khi nhận ảnh: {e}")
        return jsonify({"status": "error", "message": str(e)}), 500

# --- 8. HÀM HIỂN THỊ PREVIEW (TÙY CHỌN) ---
def display_preview():
    global latest_frame_processed
    print("Khởi động cửa sổ xem trước (Nhấn 'q' để đóng)...")
    window_name = "Posture Detection Preview"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL) # Cho phép thay đổi kích thước cửa sổ

    while True:
        frame_to_show = None # Tạo biến cục bộ để tránh lỗi race condition
        if latest_frame_processed is not None:
             # Tạo bản sao ngay lập tức để tránh bị luồng khác ghi đè
            with threading.Lock(): # Khóa tạm thời để đọc frame an toàn
                 frame_to_show = latest_frame_processed.copy()

        if frame_to_show is not None:
            # Vẽ trạng thái lên ảnh
            status_text = "OK" if posture_status == 0 else "WRONG" if posture_status == 1 else "N/A"
            color = (0, 255, 0) if posture_status == 0 else (0, 0, 255) if posture_status == 1 else (255, 0, 0)
            cv2.putText(frame_to_show, f"Status: {status_text}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            if alert_sent_for_current_session:
                 cv2.putText(frame_to_show, "ALERT SENT!", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

            cv2.imshow(window_name, frame_to_show)
        else:
            # Hiển thị ảnh chờ nếu chưa có frame
            waiting_img = np.zeros((INPUT_HEIGHT, INPUT_WIDTH, 3), dtype=np.uint8)
            cv2.putText(waiting_img, "Waiting for image...", (10, INPUT_HEIGHT // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
            cv2.imshow(window_name, waiting_img)

        # Thoát khi nhấn 'q' trên cửa sổ preview
        if cv2.waitKey(10) & 0xFF == ord('q'): # Chờ 50ms
            break

    cv2.destroyAllWindows()
    print("Đã đóng cửa sổ xem trước.")
    # Tùy chọn: Dừng cả server khi đóng cửa sổ preview
    print("Đang dừng server...")
    os._exit(0) # Dừng toàn bộ chương trình

# --- 9. KHỞI ĐỘNG SERVER ---
if __name__ == '__main__':
    # Chạy luồng xử lý nền
    processor_thread = threading.Thread(target=background_processor, daemon=True)
    processor_thread.start()
    print("✅ Luồng xử lý nền đã khởi động.")

    # Chạy luồng hiển thị preview (nếu bật)
    if SHOW_PREVIEW_WINDOW:
        display_thread = threading.Thread(target=display_preview, daemon=True)
        display_thread.start()
        print("✅ Luồng hiển thị xem trước đã khởi động.")

    # Chạy Flask server (chặn luồng chính)
    print(f"🔥 Server Flask đang chạy trên http://localhost:8000")
    print("   Cloudflare Tunnel nên được trỏ đến địa chỉ này.")
    print("   Nhấn Ctrl+C để dừng server.")
    # Chạy Flask bằng waitress thay vì server dev để ổn định hơn
    try:
        from waitress import serve
        serve(app, host='0.0.0.0', port=8000, threads=8) # threads=8 là ví dụ
    except ImportError:
        print("\n--- Cảnh báo: waitress chưa được cài đặt, đang dùng server dev của Flask ---")
        print("   Để ổn định hơn, hãy cài: pip install waitress")
        app.run(host='0.0.0.0', port=8000, debug=False)

Đang tải mô hình...
-> Mô hình YOLO, Scaler, và Classifier đã tải thành công.
   -> Đang sử dụng GPU: NVIDIA GeForce RTX 4050 Laptop GPU
✅ Luồng xử lý nền đã khởi động.
Khởi động cửa sổ xem trước (Nhấn 'q' để đóng)...
✅ Luồng hiển thị xem trước đã khởi động.
🔥 Server Flask đang chạy trên http://localhost:8000
   Cloudflare Tunnel nên được trỏ đến địa chỉ này.
   Nhấn Ctrl+C để dừng server.
-> Đã lưu ảnh đúng: saved_images/correct/correct_20251026_134030.jpg
[13:43:27] !!! GỬI CẢNH BÁO TƯ THẾ SAI !!!
-> Đã lưu ảnh sai: saved_images/incorrect/incorrect_20251026_134327.jpg
[13:44:26] Ngồi đúng đủ lâu, reset cờ cảnh báo.


: 

In [1]:
# Yêu cầu: pip install Flask opencv-python ultralytics scikit-learn joblib numpy waitress
from flask import Flask, request, jsonify
import cv2
import numpy as np
from ultralytics import YOLO
import joblib
import threading
import time
import os
from queue import Queue, Full, Empty 
import torch

# --- 1. CẤU HÌNH PATH & SIZE ---
YOLO_MODEL_PATH = 'yolo11n-pose.pt'
CLASSIFIER_PATH = 'posture_classifier.joblib'
SCALER_PATH = 'posture_scaler.joblib'

# Kích thước ảnh đầu vào (PHẢI GIỐNG LÚC TRAIN & ESP32)
INPUT_WIDTH = 320
INPUT_HEIGHT = 240
TARGET_SIZE = (INPUT_WIDTH, INPUT_HEIGHT)

# Keypoints quan trọng (PHẢI GIỐNG LÚC TRAIN)
IMPORTANT_KEYPOINTS_INDICES = [0, 5, 6, 11, 12] # Mũi, Vai trái/phải, Hông trái/phải
EXPECTED_FEATURE_LENGTH = (len(IMPORTANT_KEYPOINTS_INDICES) - (1 if IMPORTANT_KEYPOINTS_INDICES[0] == 0 else 0) ) * 2

# CÁC ĐƯỜNG NỐI KHUNG XƯƠNG (CHỈ DÙNG KHI LƯU ẢNH)
SKELETON_CONNECTIONS = [
    (0, 5), (0, 6),    # Mũi -> Vai
    (5, 6), (11, 12),  # Vai -> Vai, Hông -> Hông
    (5, 11), (6, 12),  # Vai -> Hông
    (5, 7), (6, 8),    # Tay trên (Vai -> Khuỷu)
    (11, 13), (12, 14) # Chân trên (Hông -> Gối)
]

# --- CẤU HÌNH THÔNG BÁO & LƯU TRỮ ---
INCORRECT_POSTURE_THRESHOLD_SECONDS = 60 # Cảnh báo sau 60 giây sai
CORRECT_POSTURE_RESET_ALERT_SECONDS = 30 # Reset cảnh báo sau 30 giây đúng
SAVE_IMAGE_ON_ALERT = True
SAVE_PATH_CORRECT = "saved_images/correct/"
SAVE_PATH_INCORRECT = "saved_images/incorrect/"
SHOW_PREVIEW_WINDOW = True # Hiển thị cửa sổ OpenCV trên laptop

# Tạo thư mục lưu ảnh
os.makedirs(SAVE_PATH_CORRECT, exist_ok=True)
os.makedirs(SAVE_PATH_INCORRECT, exist_ok=True)

# --- 2. TẢI MODEL ---
print("Đang tải mô hình...")
try:
    yolo_model = YOLO(YOLO_MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    classifier = joblib.load(CLASSIFIER_PATH)
    print("-> Mô hình YOLO, Scaler, và Classifier đã tải thành công.")
    if torch.cuda.is_available(): print(f"   -> Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
    else: print("   -> ⚠️ Cảnh báo: Không tìm thấy GPU, đang sử dụng CPU.")
except Exception as e:
    print(f"❌ Lỗi tải model: {e}")
    exit()

# --- 3. KHỞI TẠO FLASK VÀ CÁC BIẾN TOÀN CỤC ---
app = Flask(__name__)
frame_queue = Queue(maxsize=1) 

# Biến theo dõi trạng thái
latest_frame_processed = None
latest_keypoints = None # Biến lưu keypoints cho việc vẽ (chỉ dùng khi lưu ảnh)
posture_status = 2 
incorrect_posture_start_time = None
alert_sent_for_current_session = False
correct_posture_start_time = None
last_saved_correct_image_time = 0

# --- 4. HÀM TRÍCH XUẤT ĐẶC TRƯNG TỪ FRAME ---
def extract_pose_features_from_frame(frame, yolo_model):
    """Trích xuất và chuẩn hóa keypoints từ frame ảnh."""
    try:
        if frame is None: return None, None
        img_resized = frame

        results = yolo_model(img_resized, verbose=False)
        keypoints_data = results[0].keypoints 
        
        if results and keypoints_data and keypoints_data.shape[1] > 0:
            keypoints_xy = keypoints_data.xy[0].cpu().numpy()
            features = []
            for idx in IMPORTANT_KEYPOINTS_INDICES:
                if idx < len(keypoints_xy) and keypoints_xy[idx][0] > 1 and keypoints_xy[idx][1] > 1:
                    features.extend(keypoints_xy[idx])
                else:
                    features.extend([0, 0])

            # Chuẩn hóa theo mũi
            if len(features) > 1 and IMPORTANT_KEYPOINTS_INDICES[0] == 0 and features[0] != 0 and features[1] != 0:
                 nose_x, nose_y = features[0], features[1]
                 normalized_features = []
                 for i in range(0, len(features), 2):
                      normalized_features.append(features[i] - nose_x if features[i] != 0 else 0)
                      normalized_features.append(features[i+1] - nose_y if features[i+1] != 0 else 0)
                 final_features = np.array(normalized_features[2:])
                 return final_features if len(final_features) == EXPECTED_FEATURE_LENGTH else None, keypoints_data
            else: 
                 return None, keypoints_data 
        else: return None, None
    except Exception as e:
        return None, None

# --- 5. HÀM DỰ ĐOÁN TƯ THẾ ---
def predict_posture(frame, yolo_model, scaler, classifier):
    """Trích xuất features, chuẩn hóa, dự đoán tư thế VÀ trả về keypoints."""
    features, keypoints_data = extract_pose_features_from_frame(frame, yolo_model)
    status_code = 2
    
    if features is not None:
        try:
            features_reshaped = features.reshape(1, -1)
            features_scaled = scaler.transform(features_reshaped)
            prediction = classifier.predict(features_scaled)[0]
            status_code = int(prediction) # 0 hoặc 1
        except Exception as e_svm:
            status_code = 2 
    
    return status_code, keypoints_data

# --- 6. HÀM VẼ (CHỈ DÙNG CHO LƯU ẢNH) ---
def draw_pose(image, keypoints_obj):
    """Vẽ keypoints và skeleton lên ảnh."""
    if keypoints_obj is None or keypoints_obj.shape[1] == 0:
        return image 

    keypoints_xy = keypoints_obj.xy[0].cpu().numpy().astype(int)
    keypoints_conf = None
    if keypoints_obj.conf is not None:
         keypoints_conf = keypoints_obj.conf[0].cpu().numpy()

    # Vẽ các keypoints
    for i in IMPORTANT_KEYPOINTS_INDICES:
        if i < len(keypoints_xy):
            x, y = keypoints_xy[i]
            if x > 0 and y > 0:
                color = (0, 255, 0)
                cv2.circle(image, (x, y), 3, color, -1) 

    # Vẽ các đường nối skeleton
    for i, j in SKELETON_CONNECTIONS:
        if i < len(keypoints_xy) and j < len(keypoints_xy):
            pt1 = tuple(keypoints_xy[i])
            pt2 = tuple(keypoints_xy[j])
            if pt1[0] > 0 and pt1[1] > 0 and pt2[0] > 0 and pt2[1] > 0:
                cv2.line(image, pt1, pt2, (255, 255, 0), 1) 

    return image


def draw_labeled_frame_for_save(frame, keypoints, current_status, is_alert):
    """Vẽ keypoints, skeleton và status lên ảnh trước khi lưu."""
    
    image_to_save = frame.copy()
    image_to_save = draw_pose(image_to_save, keypoints)
    
    status_text = "CORRECT" if current_status == 0 else "INCORRECT" if current_status == 1 else "NO_PERSON"
    alert_text = "ALERT TRIGGERED" if is_alert else ""
    
    color = (0, 255, 0) if current_status == 0 else (0, 0, 255)
    
    cv2.putText(image_to_save, f"Saved Status: {status_text}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    if is_alert:
        cv2.putText(image_to_save, alert_text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    return image_to_save

# --- 9. LUỒNG XỬ LÝ ẢNH NỀN ---
def background_processor():
    global latest_frame_processed, posture_status, latest_keypoints
    global incorrect_posture_start_time, alert_sent_for_current_session
    global correct_posture_start_time, last_saved_correct_image_time

    while True:
        try:
            frame_to_process = frame_queue.get(block=True)
            current_status, current_keypoints = predict_posture(frame_to_process, yolo_model, scaler, classifier)
            
            posture_status = current_status
            
            # Cập nhật frame và keypoints cho luồng hiển thị (AN TOÀN)
            with threading.Lock():
                latest_frame_processed = frame_to_process.copy()
                latest_keypoints = current_keypoints 
            
            # --- Logic Cảnh báo theo thời gian ---
            current_time = time.time()
            if current_status == 1: # Sai
                correct_posture_start_time = None
                if incorrect_posture_start_time is None: incorrect_posture_start_time = current_time
                else:
                    duration_incorrect = current_time - incorrect_posture_start_time
                    if duration_incorrect >= INCORRECT_POSTURE_THRESHOLD_SECONDS and not alert_sent_for_current_session:
                        print(f"[{time.strftime('%H:%M:%S')}] !!! GỬI CẢNH BÁO TƯ THẾ SAI !!!")
                        alert_sent_for_current_session = True
                        if SAVE_IMAGE_ON_ALERT:
                            timestamp = time.strftime("%Y%m%d_%H%M%S")
                            filename = os.path.join(SAVE_PATH_INCORRECT, f"incorrect_{timestamp}.jpg")
                            labeled_frame = draw_labeled_frame_for_save(frame_to_process, current_keypoints, current_status, True)
                            try: cv2.imwrite(filename, labeled_frame); print(f"-> Đã lưu ảnh sai: {filename}")
                            except Exception as e_save: print(f"Lỗi lưu ảnh sai: {e_save}")

            elif current_status == 0: # Đúng
                incorrect_posture_start_time = None
                if correct_posture_start_time is None: correct_posture_start_time = current_time
                else:
                    duration_correct = current_time - correct_posture_start_time
                    if alert_sent_for_current_session and duration_correct >= CORRECT_POSTURE_RESET_ALERT_SECONDS:
                        print(f"[{time.strftime('%H:%M:%S')}] Ngồi đúng đủ lâu, reset cờ cảnh báo.")
                        alert_sent_for_current_session = False
                if SAVE_IMAGE_ON_ALERT and (current_time - last_saved_correct_image_time > 300): # Lưu ảnh đúng mỗi 5 phút
                     timestamp = time.strftime("%Y%m%d_%H%M%S")
                     filename = os.path.join(SAVE_PATH_CORRECT, f"correct_{timestamp}.jpg")
                     labeled_frame = draw_labeled_frame_for_save(frame_to_process, current_keypoints, current_status, False)
                     try: cv2.imwrite(filename, labeled_frame); print(f"-> Đã lưu ảnh đúng: {filename}"); last_saved_correct_image_time = current_time
                     except Exception as e_save: print(f"Lỗi lưu ảnh đúng: {e_save}")

            else: # Không phát hiện
                incorrect_posture_start_time = None
                correct_posture_start_time = None
                if alert_sent_for_current_session:
                     print(f"[{time.strftime('%H:%M:%S')}] Không thấy người, reset cờ cảnh báo.")
                     alert_sent_for_current_session = False

            frame_queue.task_done()
        except Exception as e:
            print(f"Lỗi trong luồng xử lý nền: {e}")
            posture_status = 2
            time.sleep(1)

# --- 7. ROUTE NHẬN ẢNH TỪ ESP32/CLIENT ---
@app.route('/', methods=['POST'])
def receive_image():
    try:
        img_bytes = request.data
        if not img_bytes:
            return jsonify({"status": "error", "message": "No image data received"}), 400

        nparr = np.frombuffer(img_bytes, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if frame is not None:
            # ===> LẬT ẢNH DỌC <===
            frame = cv2.flip(frame, 0)
            
            if frame.shape[1] != INPUT_WIDTH or frame.shape[0] != INPUT_HEIGHT:
                 frame = cv2.resize(frame, TARGET_SIZE) 

            # Đưa frame vào queue (luôn lấy frame mới nhất)
            try:
                try: frame_queue.get_nowait()
                except Empty: pass
                frame_queue.put_nowait(frame)
            except Full: pass
                
            # Trả về trạng thái cảnh báo cuối cùng
            status_to_send_esp = 1 if alert_sent_for_current_session else posture_status
            message = "GOOD_POSTURE" if status_to_send_esp == 0 else "BAD_POSTURE_ALERT" if status_to_send_esp == 1 else "NO_PERSON/ERROR"

            return jsonify({"status_code": status_to_send_esp, "message": message}), 200
        else:
            return jsonify({"status": "error", "message": "Lỗi giải mã JPEG"}), 400
    except Exception as e:
        print(f"Lỗi server khi nhận ảnh: {e}")
        return jsonify({"status": "error", "message": str(e)}), 500

# --- 8. HÀM HIỂN THỊ PREVIEW (CHỈ VẼ TEXT LÊN FRAME THÔ) ---
def display_preview():
    global latest_frame_processed
    print("Khởi động cửa sổ xem trước (Nhấn 'q' để đóng)...")
    window_name = "Posture Detection Preview"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL) 

    while True:
        frame_to_show = None
        
        with threading.Lock():
            if latest_frame_processed is not None:
                frame_to_show = latest_frame_processed.copy()

        if frame_to_show is not None:
            # CHỈ VẼ TEXT (RẤT NHANH)
            status_text = "OK" if posture_status == 0 else "WRONG" if posture_status == 1 else "N/A"
            color = (0, 255, 0) if posture_status == 0 else (0, 0, 255) if posture_status == 1 else (255, 0, 0)
            cv2.putText(frame_to_show, f"Status: {status_text}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            if alert_sent_for_current_session:
                 cv2.putText(frame_to_show, "ALERT SENT!", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

            cv2.imshow(window_name, frame_to_show)
        else:
            # Hiển thị ảnh chờ
            waiting_img = np.zeros((INPUT_HEIGHT, INPUT_WIDTH, 3), dtype=np.uint8)
            cv2.putText(waiting_img, "Waiting for image...", (10, INPUT_HEIGHT // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
            cv2.imshow(window_name, waiting_img)

        # Chờ 10ms (100 FPS max)
        if cv2.waitKey(10) & 0xFF == ord('q'): 
            break

    cv2.destroyAllWindows()
    print("Đã đóng cửa sổ xem trước.")
    os._exit(0) 

# --- 9. KHỞI ĐỘNG SERVER ---
if __name__ == '__main__':
    # Chạy luồng xử lý nền
    processor_thread = threading.Thread(target=background_processor, daemon=True)
    processor_thread.start()
    print("✅ Luồng xử lý nền đã khởi động.")

    # Chạy luồng hiển thị preview (nếu bật)
    if SHOW_PREVIEW_WINDOW:
        display_thread = threading.Thread(target=display_preview, daemon=True)
        display_thread.start()
        print("✅ Luồng hiển thị xem trước đã khởi động.")

    # Chạy Flask server (chặn luồng chính)
    print(f"🔥 Server Flask đang chạy trên http://localhost:8000")
    print("   Nhấn Ctrl+C để dừng server.")
    try:
        from waitress import serve
        serve(app, host='0.0.0.0', port=8000, threads=8)
    except ImportError:
        print("\n--- Cảnh báo: waitress chưa được cài đặt, đang dùng server dev của Flask ---")
        app.run(host='0.0.0.0', port=8000, debug=False)

ModuleNotFoundError: No module named 'flask'

In [3]:
# Yêu cầu: pip install Flask opencv-python ultralytics scikit-learn joblib numpy waitress torch torchvision
from flask import Flask, request, jsonify
import cv2
import numpy as np
from ultralytics import YOLO
import joblib
import threading
import time
import os
from queue import Queue, Full, Empty # Đảm bảo import Empty và Full
import torch

# --- 1. CẤU HÌNH PATH & SIZE ---
YOLO_MODEL_PATH = 'yolo11n-pose.pt'
CLASSIFIER_PATH = 'posture_classifier.joblib'
SCALER_PATH = 'posture_scaler.joblib'

# Kích thước ảnh đầu vào (PHẢI GIỐNG LÚC TRAIN & ESP32)
INPUT_WIDTH = 320
INPUT_HEIGHT = 240
TARGET_SIZE = (INPUT_WIDTH, INPUT_HEIGHT)

# Keypoints quan trọng (PHẢI GIỐNG LÚC TRAIN)
IMPORTANT_KEYPOINTS_INDICES = [0, 5, 6, 11, 12] 
EXPECTED_FEATURE_LENGTH = (len(IMPORTANT_KEYPOINTS_INDICES) - (1 if IMPORTANT_KEYPOINTS_INDICES[0] == 0 else 0) ) * 2

# ✅ KEYPOINTS ĐỂ KIỂM TRA TRẠNG THÁI ĐỨNG (Hips and Knees) - COCO Indices:
# 11: Left Hip, 12: Right Hip, 13: Left Knee, 14: Right Knee
STANDING_CHECK_INDICES = [11, 12, 13, 14]
MIN_STANDING_KEYPOINTS = 3 # Yêu cầu ít nhất 3/4 điểm (ví dụ: 2 hông + 1 gối) để xác nhận đứng

# ✅ SỐ LƯỢNG KEYPOINTS HỢP LỆ TỐI THIỂU (Thử giảm xuống 4 hoặc 3)
MIN_VALID_KEYPOINTS_FOR_PRESENCE = 4 # Hoặc thử 3 nếu 4 vẫn lỗi

# CÁC ĐƯỜNG NỐI KHUNG XƯƠNG (CHỈ DÙNG KHI LƯU ẢNH)
SKELETON_CONNECTIONS = [
    (0, 5), (0, 6),    
    (5, 6), (11, 12),  
    (5, 11), (6, 12),  
    (5, 7), (6, 8),    
    (11, 13), (12, 14) 
]

# --- CẤU HÌNH THÔNG BÁO, LƯU TRỮ & TÍNH THỜI GIAN ---
INCORRECT_POSTURE_THRESHOLD_SECONDS = 60 # Cảnh báo sau 60 giây sai
CORRECT_POSTURE_RESET_ALERT_SECONDS = 30 # Reset cảnh báo sau 30 giây đúng
SAVE_IMAGE_ON_ALERT = True
SAVE_PATH_CORRECT = "saved_images/correct/"
SAVE_PATH_INCORRECT = "saved_images/incorrect/"
SHOW_PREVIEW_WINDOW = True 

# Tạo thư mục lưu ảnh
os.makedirs(SAVE_PATH_CORRECT, exist_ok=True)
os.makedirs(SAVE_PATH_INCORRECT, exist_ok=True)

# --- 2. TẢI MODEL ---
print("Đang tải mô hình...")
try:
    yolo_model = YOLO(YOLO_MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    classifier = joblib.load(CLASSIFIER_PATH)
    print("-> Mô hình YOLO, Scaler, và Classifier đã tải thành công.")
    if torch.cuda.is_available(): print(f"   -> Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
    else: print("   -> ⚠️ Cảnh báo: Không tìm thấy GPU, đang sử dụng CPU.")
except Exception as e:
    print(f"❌ Lỗi tải model: {e}")
    exit()

# --- 3. KHỞI TẠO FLASK VÀ CÁC BIẾN TOÀN CỤC ---
app = Flask(__name__)
frame_queue = Queue(maxsize=1) 

# Biến theo dõi trạng thái
latest_frame_processed = None
latest_keypoints = None 
posture_status = 2 
incorrect_posture_start_time = None
alert_sent_for_current_session = False
correct_posture_start_time = None
last_saved_correct_image_time = 0

# Biến tính thời gian hoạt động (Seconds)
TIME_PER_FRAME_SEC = 1.0 # Vì ESP32 gửi 1 FPS
TOTAL_SITTING_TIME_SEC = 0
TOTAL_MOVING_TIME_SEC = 0

# ✅ Biến lưu vị trí Y trung bình của hông ở frame trước
previous_hip_y = None 
VERTICAL_MOVEMENT_THRESHOLD = 30 # Ngưỡng thay đổi Y (pixels) để coi là đứng lên (cần tinh chỉnh)

# --- 4. HÀM TRÍCH XUẤT ĐẶC TRƯNG TỪ FRAME ---
def extract_pose_features_from_frame(frame, yolo_model):
    # ... (Logic trích xuất đặc trưng và chuẩn hóa giữ nguyên) ...
    try:
        if frame is None: return None, None
        img_resized = frame

        results = yolo_model(img_resized, verbose=False)
        keypoints_data = results[0].keypoints 
        
        if results and keypoints_data and keypoints_data.shape[1] > 0:
            keypoints_xy = keypoints_data.xy[0].cpu().numpy()
            features = []
            for idx in IMPORTANT_KEYPOINTS_INDICES:
                if idx < len(keypoints_xy) and keypoints_xy[idx][0] > 1 and keypoints_xy[idx][1] > 1:
                    features.extend(keypoints_xy[idx])
                else:
                    features.extend([0, 0])

            # Chuẩn hóa theo mũi
            if len(features) > 1 and IMPORTANT_KEYPOINTS_INDICES[0] == 0 and features[0] != 0 and features[1] != 0:
                 nose_x, nose_y = features[0], features[1]
                 normalized_features = []
                 for i in range(0, len(features), 2):
                      normalized_features.append(features[i] - nose_x if features[i] != 0 else 0)
                      normalized_features.append(features[i+1] - nose_y if features[i+1] != 0 else 0)
                 final_features = np.array(normalized_features[2:])
                 return final_features if len(final_features) == EXPECTED_FEATURE_LENGTH else None, keypoints_data
            elif len(features) > 0:
                 final_features = np.array(features)
                 return final_features if len(final_features) == EXPECTED_FEATURE_LENGTH else None, keypoints_data
            else: return None, None
        else: return None, None
    except Exception as e:
        return None, None

# --- 5. HÀM DỰ ĐOÁN TƯ THẾ (Thêm So sánh Vị trí Y) ---
def predict_posture(frame, yolo_model, scaler, classifier):
    global previous_hip_y # Khai báo sử dụng biến global

    keypoints_data = None
    status_code = 2 # Mặc định là vắng mặt/lỗi
    current_hip_y = None # Vị trí Y hông hiện tại

    try:
        if frame is None: return status_code, None
        img_resized = frame
        results = yolo_model(img_resized, verbose=False)
        keypoints_data = results[0].keypoints

        # 1. KIỂM TRA SỰ HIỆN DIỆN VÀ TÍNH TOÁN VỊ TRÍ HÔNG
        valid_raw_keypoints_count = 0
        hip_y_values = []
        if results and keypoints_data and keypoints_data.shape[1] > 0:
            raw_keypoints_xy = keypoints_data.xy[0].cpu().numpy().astype(int)
            
            # Đếm keypoint quan trọng và lấy vị trí Y của hông
            for idx in IMPORTANT_KEYPOINTS_INDICES: # [0, 5, 6, 11, 12]
                if idx < len(raw_keypoints_xy) and raw_keypoints_xy[idx, 0] > 1 and raw_keypoints_xy[idx, 1] > 1:
                    valid_raw_keypoints_count += 1
                    # Lưu vị trí Y của hông (index 11 và 12)
                    if idx == 11 or idx == 12:
                        hip_y_values.append(raw_keypoints_xy[idx, 1])

            # Tính vị trí Y trung bình của hông (nếu có)
            if hip_y_values:
                current_hip_y = np.mean(hip_y_values)

        # Nếu số keypoint thô quá ít -> Vắng mặt
        MIN_RAW_KEYPOINTS_FOR_SITTING = 3 
        if valid_raw_keypoints_count < MIN_RAW_KEYPOINTS_FOR_SITTING:
            previous_hip_y = None # Reset vị trí cũ khi không thấy người
            return 2, keypoints_data 

        # 2. ✅ KIỂM TRA SỰ THAY ĐỔI VỊ TRÍ Y ĐỂ PHÁT HIỆN ĐỨNG LÊN
        if previous_hip_y is not None and current_hip_y is not None:
            y_change = previous_hip_y - current_hip_y # Vị trí Y giảm (đi lên) là số dương
            # print(f"Debug Y Change: Prev={previous_hip_y:.1f}, Curr={current_hip_y:.1f}, Change={y_change:.1f}") # Bỏ comment để debug
            if y_change > VERTICAL_MOVEMENT_THRESHOLD: 
                print(f"[{time.strftime('%H:%M:%S')}] Phát hiện đứng lên (Y change: {y_change:.1f})")
                previous_hip_y = current_hip_y # Cập nhật vị trí mới
                return 2, keypoints_data # Status 2 (Standing/Moving)

        # 3. CHẠY SVM (NẾU KHÔNG ĐỨNG LÊN VÀ ĐỦ KEYPOINT)
        features = extract_and_normalize_features(keypoints_data) # Gọi hàm trích xuất đã tách

        if features is not None:
            if np.sum(features) == 0: return 2, keypoints_data # Ghost detection

            try:
                features_reshaped = features.reshape(1, -1)
                features_scaled = scaler.transform(features_reshaped)
                prediction = classifier.predict(features_scaled)[0]
                status_code = int(prediction) # 0 or 1
            except Exception as e_svm:
                status_code = 2 
        else:
             status_code = 2 

    except Exception as e_yolo:
        status_code = 2 
        previous_hip_y = None # Reset nếu có lỗi

    # Cập nhật vị trí Y của hông cho lần kiểm tra sau (chỉ khi có người)
    if current_hip_y is not None:
        previous_hip_y = current_hip_y
    # Nếu không phát hiện hông (current_hip_y is None), giữ nguyên previous_hip_y để so sánh lần sau

    return status_code, keypoints_data

# --- HÀM MỚI/SỬA ĐỔI: Chỉ trích xuất và chuẩn hóa ---
def extract_and_normalize_features(keypoints_data):
    """Chỉ trích xuất và chuẩn hóa vector đặc trưng từ keypoints đã có."""
    try:
        if keypoints_data is None or keypoints_data.shape[1] == 0: return None
        
        keypoints_xy = keypoints_data.xy[0].cpu().numpy()
        features = []
        for idx in IMPORTANT_KEYPOINTS_INDICES:
            if idx < len(keypoints_xy) and keypoints_xy[idx][0] > 1 and keypoints_xy[idx][1] > 1:
                features.extend(keypoints_xy[idx])
            else:
                features.extend([0, 0])

        # Chuẩn hóa theo mũi
        if len(features) > 1 and IMPORTANT_KEYPOINTS_INDICES[0] == 0 and features[0] != 0 and features[1] != 0:
             nose_x, nose_y = features[0], features[1]
             normalized_features = []
             for i in range(0, len(features), 2):
                  normalized_features.append(features[i] - nose_x if features[i] != 0 else 0)
                  normalized_features.append(features[i+1] - nose_y if features[i+1] != 0 else 0)
             final_features = np.array(normalized_features[2:])
             return final_features if len(final_features) == EXPECTED_FEATURE_LENGTH else None
        else: 
             return None # Không chuẩn hóa được nếu thiếu mũi
             
    except Exception as e:
        # print(f"Lỗi chuẩn hóa features: {e}") # Bỏ comment để debug
        return None
    
# --- 6. HÀM VẼ (CHỈ DÙNG CHO LƯU ẢNH) & LUỒNG XỬ LÝ ẢNH NỀN ---

def draw_pose(image, keypoints_obj):
    """Vẽ keypoints và skeleton lên ảnh."""
    if keypoints_obj is None or keypoints_obj.shape[1] == 0:
        return image 
    # ... (Logic vẽ keypoints và lines lên ảnh) ...
    keypoints_xy = keypoints_obj.xy[0].cpu().numpy().astype(int)
    keypoints_conf = None
    if keypoints_obj.conf is not None: keypoints_conf = keypoints_obj.conf[0].cpu().numpy()

    # Vẽ các keypoints
    for i in IMPORTANT_KEYPOINTS_INDICES:
        if i < len(keypoints_xy):
            x, y = keypoints_xy[i]
            if x > 0 and y > 0:
                color = (0, 255, 0)
                cv2.circle(image, (x, y), 3, color, -1) 

    # Vẽ các đường nối skeleton
    for i, j in SKELETON_CONNECTIONS:
        if i < len(keypoints_xy) and j < len(keypoints_xy):
            pt1 = tuple(keypoints_xy[i])
            pt2 = tuple(keypoints_xy[j])
            if pt1[0] > 0 and pt1[1] > 0 and pt2[0] > 0 and pt2[1] > 0:
                cv2.line(image, pt1, pt2, (255, 255, 0), 1) 
    return image

def draw_labeled_frame_for_save(frame, keypoints, current_status, is_alert):
    """Vẽ keypoints, skeleton và status lên ảnh trước khi lưu."""
    image_to_save = frame.copy()
    image_to_save = draw_pose(image_to_save, keypoints)
    
    status_text = "CORRECT" if current_status == 0 else "INCORRECT" if current_status == 1 else "NO_PERSON"
    alert_text = "ALERT TRIGGERED" if is_alert else ""
    
    color = (0, 255, 0) if current_status == 0 else (0, 0, 255)
    
    cv2.putText(image_to_save, f"Saved Status: {status_text}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    if is_alert:
        cv2.putText(image_to_save, alert_text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    return image_to_save

# --- LUỒNG XỬ LÝ ẢNH NỀN (TÍCH HỢP TÍNH THỜI GIAN) ---
def background_processor():
    global latest_frame_processed, posture_status, latest_keypoints
    global incorrect_posture_start_time, alert_sent_for_current_session
    global correct_posture_start_time, last_saved_correct_image_time
    global TOTAL_SITTING_TIME_SEC, TOTAL_MOVING_TIME_SEC # ✅ Biến tính thời gian

    while True:
        try:
            frame_to_process = frame_queue.get(block=True)
            current_status, current_keypoints = predict_posture(frame_to_process, yolo_model, scaler, classifier)
            
            # Cập nhật trạng thái và frame/keypoints cho hiển thị
            posture_status = current_status
            with threading.Lock(): 
                latest_frame_processed = frame_to_process.copy()
                latest_keypoints = current_keypoints
            
            # --- LOGIC TÍNH THỜI GIAN VÀ CẢNH BÁO ---
            current_time = time.time()
            
            if current_status == 0 or current_status == 1: # Ngồi (Đúng hoặc Sai)
                TOTAL_SITTING_TIME_SEC += TIME_PER_FRAME_SEC
                
                # Logic Cảnh báo ngồi sai (status 1)
                if current_status == 1: 
                    correct_posture_start_time = None
                    if incorrect_posture_start_time is None: incorrect_posture_start_time = current_time
                    else:
                        duration_incorrect = current_time - incorrect_posture_start_time
                        if duration_incorrect >= INCORRECT_POSTURE_THRESHOLD_SECONDS and not alert_sent_for_current_session:
                            print(f"[{time.strftime('%H:%M:%S')}] !!! GỬI CẢNH BÁO TƯ THẾ SAI !!!")
                            alert_sent_for_current_session = True
                            if SAVE_IMAGE_ON_ALERT:
                                timestamp = time.strftime("%Y%m%d_%H%M%S")
                                filename = os.path.join(SAVE_PATH_INCORRECT, f"incorrect_{timestamp}.jpg")
                                labeled_frame = draw_labeled_frame_for_save(frame_to_process, current_keypoints, current_status, True)
                                try: cv2.imwrite(filename, labeled_frame); print(f"-> Đã lưu ảnh sai: {filename}")
                                except Exception as e_save: print(f"Lỗi lưu ảnh sai: {e_save}")
                
                # Logic Reset cảnh báo (khi ngồi đúng)
                elif current_status == 0: 
                    incorrect_posture_start_time = None
                    if correct_posture_start_time is None: correct_posture_start_time = current_time
                    else:
                        duration_correct = current_time - correct_posture_start_time
                        if alert_sent_for_current_session and duration_correct >= CORRECT_POSTURE_RESET_ALERT_SECONDS:
                            print(f"[{time.strftime('%H:%M:%S')}] Ngồi đúng đủ lâu, reset cờ cảnh báo.")
                            alert_sent_for_current_session = False
                    if SAVE_IMAGE_ON_ALERT and (current_time - last_saved_correct_image_time > 300): # Lưu ảnh đúng mỗi 5 phút
                         timestamp = time.strftime("%Y%m%d_%H%M%S")
                         filename = os.path.join(SAVE_PATH_CORRECT, f"correct_{timestamp}.jpg")
                         labeled_frame = draw_labeled_frame_for_save(frame_to_process, current_keypoints, current_status, False)
                         try: cv2.imwrite(filename, labeled_frame); print(f"-> Đã lưu ảnh đúng: {filename}"); last_saved_correct_image_time = current_time
                         except Exception as e_save: print(f"Lỗi lưu ảnh đúng: {e_save}")


            else: # current_status == 2 (Không phát hiện)
                TOTAL_MOVING_TIME_SEC += TIME_PER_FRAME_SEC # Tính là thời gian vận động
                incorrect_posture_start_time = None
                correct_posture_start_time = None
                if alert_sent_for_current_session:
                     print(f"[{time.strftime('%H:%M:%S')}] Không thấy người, reset cờ cảnh báo.")
                     alert_sent_for_current_session = False

            frame_queue.task_done()
        except Exception as e:
            print(f"Lỗi trong luồng xử lý nền: {e}")
            posture_status = 2
            time.sleep(1)

# --- 7. ROUTE NHẬN ẢNH TỪ ESP32/CLIENT (TRẢ VỀ THỜI GIAN) ---
@app.route('/', methods=['POST'])
def receive_image():
    try:
        img_bytes = request.data
        if not img_bytes:
            return jsonify({"status": "error", "message": "No image data received"}), 400

        nparr = np.frombuffer(img_bytes, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if frame is not None:
            # Lật ảnh dọc để sửa lỗi lắp đặt camera
            frame = cv2.flip(frame, 0)
            
            if frame.shape[1] != INPUT_WIDTH or frame.shape[0] != INPUT_HEIGHT:
                 frame = cv2.resize(frame, TARGET_SIZE) 

            try:
                try: frame_queue.get_nowait()
                except Empty: pass
                frame_queue.put_nowait(frame)
            except Full: pass
                
            # Trả về trạng thái cảnh báo cuối cùng VÀ THÔNG SỐ THỜI GIAN
            status_to_send_esp = 1 if alert_sent_for_current_session else posture_status
            message = "GOOD_POSTURE" if status_to_send_esp == 0 else "BAD_POSTURE_ALERT" if status_to_send_esp == 1 else "NO_PERSON/ERROR"

            return jsonify({
                "status_code": status_to_send_esp,
                "message": message,
                "sitting_time_min": TOTAL_SITTING_TIME_SEC / 60,
                "moving_time_min": TOTAL_MOVING_TIME_SEC / 60
            }), 200
        else:
            return jsonify({"status": "error", "message": "Lỗi giải mã JPEG"}), 400
    except Exception as e:
        print(f"Lỗi server khi nhận ảnh: {e}")
        return jsonify({"status": "error", "message": str(e)}), 500

# --- 8. HÀM HIỂN THỊ PREVIEW (TÙY CHỌN) ---
def display_preview():
    global latest_frame_processed, latest_keypoints
    global TOTAL_SITTING_TIME_SEC, TOTAL_MOVING_TIME_SEC # KHAI BÁO GLOBAL

    print("Khởi động cửa sổ xem trước (Nhấn 'q' để đóng)...")
    window_name = "Posture Detection Preview"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL) 

    while True:
        frame_to_show = None
        keypoints_to_draw = None
        
        with threading.Lock():
            if latest_frame_processed is not None:
                frame_to_show = latest_frame_processed.copy()
                keypoints_to_draw = latest_keypoints 

        if frame_to_show is not None:
            # CHỈ VẼ TEXT (RẤT NHANH)
            status_text = "OK" if posture_status == 0 else "WRONG" if posture_status == 1 else "N/A"
            color = (0, 255, 0) if posture_status == 0 else (0, 0, 255) if posture_status == 1 else (255, 0, 0)
            
            # 1. Vẽ Trạng thái (Status)
            cv2.putText(frame_to_show, f"Status: {status_text}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            
            # 2. Vẽ Cảnh báo (Alert)
            if alert_sent_for_current_session:
                 cv2.putText(frame_to_show, "ALERT SENT!", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
            
            # 3. ✅ VẼ THỜI GIAN TÍCH LŨY
            sitting_time_min = TOTAL_SITTING_TIME_SEC / 60
            moving_time_min = TOTAL_MOVING_TIME_SEC / 60

            # Hiển thị Thời gian Ngồi
            cv2.putText(frame_to_show, f"Sitting: {sitting_time_min:.1f} min", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            # Hiển thị Thời gian Vận động
            cv2.putText(frame_to_show, f"Moving: {moving_time_min:.1f} min", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)


            cv2.imshow(window_name, frame_to_show)
        else:
            # ... (code hiển thị ảnh chờ giữ nguyên) ...
            waiting_img = np.zeros((INPUT_HEIGHT, INPUT_WIDTH, 3), dtype=np.uint8)
            cv2.putText(waiting_img, "Waiting for image...", (10, INPUT_HEIGHT // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
            cv2.imshow(window_name, waiting_img)

        # Chờ 10ms (100 FPS max)
        if cv2.waitKey(10) & 0xFF == ord('q'): 
            break

    cv2.destroyAllWindows()
    print("Đã đóng cửa sổ xem trước.")
    os._exit(0)

# --- 9. KHỞI ĐỘNG SERVER ---
if __name__ == '__main__':
    # Chạy luồng xử lý nền
    processor_thread = threading.Thread(target=background_processor, daemon=True)
    processor_thread.start()
    print("✅ Luồng xử lý nền đã khởi động.")

    # Chạy luồng hiển thị preview (nếu bật)
    if SHOW_PREVIEW_WINDOW:
        display_thread = threading.Thread(target=display_preview, daemon=True)
        display_thread.start()
        print("✅ Luồng hiển thị xem trước đã khởi động.")

    # Chạy Flask server (chặn luồng chính)
    print(f"🔥 Server Flask đang chạy trên http://localhost:8000")
    print("   Nhấn Ctrl+C để dừng server.")
    try:
        from waitress import serve
        serve(app, host='0.0.0.0', port=8000, threads=8)
    except ImportError:
        print("\n--- Cảnh báo: waitress chưa được cài đặt, đang dùng server dev của Flask ---")
        app.run(host='0.0.0.0', port=8000, debug=False)

Đang tải mô hình...
-> Mô hình YOLO, Scaler, và Classifier đã tải thành công.
   -> Đang sử dụng GPU: NVIDIA GeForce RTX 4050 Laptop GPU
✅ Luồng xử lý nền đã khởi động.
Khởi động cửa sổ xem trước (Nhấn 'q' để đóng)...
✅ Luồng hiển thị xem trước đã khởi động.
🔥 Server Flask đang chạy trên http://localhost:8000
   Nhấn Ctrl+C để dừng server.


: 

In [ ]:
# Yêu cầu: pip install Flask opencv-python numpy
from flask import Flask, request, Response
import time
import cv2
import numpy as np
import threading # Cần threading để chạy hiển thị song song
import os # Thêm os để dùng os._exit
from queue import Queue, Empty # Chỉ cần Queue và Empty

# --- CẤU HÌNH ---
INPUT_WIDTH = 320
INPUT_HEIGHT = 240
TARGET_SIZE = (INPUT_WIDTH, INPUT_HEIGHT)
SHOW_PREVIEW_WINDOW = True

# --- KHỞI TẠO FLASK VÀ BIẾN TOÀN CỤC ---
app = Flask(__name__)
frame_queue = Queue(maxsize=1) # Queue chỉ giữ frame mới nhất
latest_frame_processed = None # Frame mới nhất đã xử lý (chỉ để hiển thị)
display_lock = threading.Lock() # Lock để truy cập frame an toàn

# --- ROUTE NHẬN ẢNH TỪ ESP32 ---
@app.route('/', methods=['POST'])
def receive_image():
    """Nhận dữ liệu ảnh thô, giải mã và cập nhật frame để hiển thị."""
    global latest_frame_processed
    try:
        img_bytes = request.data
        if img_bytes:
            # print(f"[{time.strftime('%H:%M:%S')}] Received image data: {len(img_bytes)} bytes") # Bỏ comment để debug

            # --- GIẢI MÃ VÀ CẬP NHẬT FRAME ---
            nparr = np.frombuffer(img_bytes, np.uint8)
            frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

            if frame is not None:
                # Lật ảnh nếu cần
                frame = cv2.flip(frame, 0)
                # Resize nếu cần
                if frame.shape[1] != INPUT_WIDTH or frame.shape[0] != INPUT_HEIGHT:
                     frame = cv2.resize(frame, TARGET_SIZE)

                # Cập nhật frame mới nhất một cách an toàn
                with display_lock:
                    latest_frame_processed = frame.copy()
                return "OK", 200
            else:
                # print(f"[{time.strftime('%H:%M:%S')}] Error decoding JPEG.") # Bỏ comment để debug
                return "JPEG Decode Error", 400
            # --- --- --- --- --- --- --- --- ---

        else:
            # print(f"[{time.strftime('%H:%M:%S')}] Received empty request.") # Bỏ comment để debug
            return "No image data received", 400

    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] Server error: {e}")
        return "Internal Server Error", 500

# --- HÀM HIỂN THỊ PREVIEW ---
def display_preview():
    """Hiển thị frame ảnh mới nhất trong cửa sổ OpenCV."""
    global latest_frame_processed
    window_name = "ESP32 Image Stream"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    
    # ✅ THÊM LỆNH NÀY: Đảm bảo cửa sổ được tạo trước vòng lặp
    cv2.waitKey(1) 

    while True:
        frame_copy = None
        # Lấy frame mới nhất một cách an toàn
        with display_lock:
            if latest_frame_processed is not None:
                frame_copy = latest_frame_processed.copy()

        if frame_copy is not None:
            cv2.imshow(window_name, frame_copy)
        else:
            # Hiển thị ảnh chờ nếu chưa có frame
            waiting_img = np.zeros((INPUT_HEIGHT, INPUT_WIDTH, 3), dtype=np.uint8)
            cv2.putText(waiting_img, "Waiting for image...", (10, INPUT_HEIGHT // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
            cv2.imshow(window_name, waiting_img)

        # Chờ 10ms và kiểm tra phím 'q' để thoát
        key = cv2.waitKey(10) & 0xFF
        if key == ord('q'):
            break
        # Kiểm tra xem cửa sổ có bị đóng không (trên một số hệ thống)
        try:
             if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                 break
        except:
             # Bỏ qua lỗi nếu getWindowProperty không được hỗ trợ
             pass


    cv2.destroyAllWindows()
    print("Preview window closed.")
    os._exit(0) # Dừng toàn bộ chương trình khi đóng cửa sổ

# Khối chính để chạy server
if __name__ == '__main__':
    PORT = 8000
    print(f"🔥 Simple Image Receiver running on http://localhost:{PORT}")
    print("   Waiting for images from ESP32...")
    print("   Press 'q' on the preview window to stop.")

    # KHỞI ĐỘNG LUỒNG HIỂN THỊ
    if SHOW_PREVIEW_WINDOW:
        display_thread = threading.Thread(target=display_preview, daemon=True)
        display_thread.start()
        print("✅ Luồng hiển thị xem trước đã khởi động.")
    else:
        print("ℹ️ Hiển thị xem trước đã bị tắt (SHOW_PREVIEW_WINDOW = False).")


    # Chạy server Flask (chặn luồng chính)
    try:
        from waitress import serve
        serve(app, host='0.0.0.0', port=PORT)
    except ImportError:
        print("\n--- Waitress not installed, using Flask's dev server ---")
        app.run(host='0.0.0.0', port=PORT, debug=False)

🔥 Simple Image Receiver running on http://localhost:8000
   Waiting for images from ESP32...
   Press 'q' on the preview window to stop.
✅ Luồng hiển thị xem trước đã khởi động.

--- Waitress not installed, using Flask's dev server ---
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://192.168.50.162:8000
Press CTRL+C to quit
127.0.0.1 - - [03/Nov/2025 16:00:56] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:01] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:05] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:10] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:15] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:19] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:24] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:29] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:33] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:37] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:41] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:45] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:52] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [03/Nov/2025 16:01:57] "POST / HTTP/1.1" 200 -
127.0.0.1 

In [1]:
# Yêu cầu: pip install Flask opencv-python ultralytics scikit-learn joblib numpy waitress torch torchvision
from flask import Flask, request, jsonify
import cv2
import numpy as np
from ultralytics import YOLO
import joblib
import threading
import time
import os
from queue import Queue, Full, Empty
import torch

# --- 1. CẤU HÌNH PATH & SIZE ---
YOLO_MODEL_PATH = 'yolo11n-pose.pt'
CLASSIFIER_PATH = 'posture_classifier_3class.joblib'
SCALER_PATH = 'posture_scaler_3class.joblib' # 🐞 SỬA LỖI: Đã xóa ký tự lạ

# Kích thước ảnh đầu vào (PHẢI GIỐNG LÚC TRAIN & ESP32)
INPUT_WIDTH = 320
INPUT_HEIGHT = 240
TARGET_SIZE = (INPUT_WIDTH, INPUT_HEIGHT)

# Keypoints quan trọng (PHẢI GIỐNG LÚC TRAIN)
IMPORTANT_KEYPOINTS_INDICES = [0, 5, 6, 11, 12] # Mũi, Vai trái/phải, Hông trái/phải
EXPECTED_FEATURE_LENGTH = (len(IMPORTANT_KEYPOINTS_INDICES) - 1) * 2 # Đã bỏ mũi
MIN_RAW_KEYPOINTS_FOR_SITTING = 3 # Ngưỡng keypoint thô tối thiểu

# --- CẤU HÌNH THÔNG BÁO, LƯU TRỮ & TÍNH THỜI GIAN ---
INCORRECT_POSTURE_THRESHOLD_SECONDS = 60 # Cảnh báo sau 60 giây sai
CORRECT_POSTURE_RESET_ALERT_SECONDS = 30 # Reset cảnh báo sau 30 giây đúng
SHOW_PREVIEW_WINDOW = True # Hiển thị cửa sổ OpenCV trên laptop

# 🐞 SỬA LỖI LOGIC THỜI GIAN
# Phải khớp với 'frameDelayMs = 1000' của ESP32
TIME_PER_FRAME_SEC = 1.0 # 1.0 giây/frame (thay vì 5.0)

# Tạo thư mục lưu ảnh
os.makedirs("saved_images/correct", exist_ok=True)
os.makedirs("saved_images/incorrect", exist_ok=True)
os.makedirs("saved_images/absent", exist_ok=True)

# --- 2. TẢI MODEL ---
print("Đang tải mô hình...")
try:
    yolo_model = YOLO(YOLO_MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    classifier = joblib.load(CLASSIFIER_PATH)
    print("-> Mô hình YOLO, Scaler (3-lớp), và Classifier (3-lớp) đã tải thành công.")
    if torch.cuda.is_available(): print(f"    -> Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")
    else: print("    -> ⚠️ Cảnh báo: Không tìm thấy GPU, đang sử dụng CPU.")
except Exception as e:
    print(f"❌ Lỗi tải model: {e}")
    exit()

# --- 3. KHỞI TẠO FLASK VÀ CÁC BIẾN TOÀN CỤC ---
app = Flask(__name__)
frame_queue = Queue(maxsize=1)

# Biến theo dõi trạng thái
latest_frame_processed = None
latest_keypoints = None
posture_status = 2 # 0: Correct, 1: Incorrect, 2: Absent
incorrect_posture_start_time = None
alert_sent_for_current_session = False
correct_posture_start_time = None

# Biến tính thời gian hoạt động (Seconds)
TOTAL_SITTING_TIME_SEC = 0
TOTAL_MOVING_TIME_SEC = 0

# --- 4. HÀM TRÍCH XUẤT ĐẶC TRƯNG ---
def extract_and_normalize_features(keypoints_data):
    """Trích xuất và chuẩn hóa vector đặc trưng từ keypoints đã có."""
    try:
        if keypoints_data is None or keypoints_data.shape[1] == 0: return None
        keypoints_xy = keypoints_data.xy[0].cpu().numpy()
        features = []
        for idx in IMPORTANT_KEYPOINTS_INDICES:
            if idx < len(keypoints_xy) and keypoints_xy[idx][0] > 1 and keypoints_xy[idx][1] > 1:
                features.extend(keypoints_xy[idx])
            else:
                features.extend([0, 0])
        if len(features) > 1 and IMPORTANT_KEYPOINTS_INDICES[0] == 0 and features[0] != 0 and features[1] != 0:
            nose_x, nose_y = features[0], features[1]
            normalized_features = []
            for i in range(0, len(features), 2):
                normalized_features.append(features[i] - nose_x if features[i] != 0 else 0)
                normalized_features.append(features[i+1] - nose_y if features[i+1] != 0 else 0)
            final_features = np.array(normalized_features[2:])
            return final_features if len(final_features) == EXPECTED_FEATURE_LENGTH else None
        else: return None
    except Exception as e: return None

# --- 5. HÀM DỰ ĐOÁN TƯ THẾ (Sử dụng model 3 lớp) ---
def predict_posture(frame, yolo_model, scaler, classifier):
    keypoints_data = None
    status_code = 2 # Mặc định: Absent
    try:
        if frame is None: return status_code, None
        img_resized = frame
        results = yolo_model(img_resized, verbose=False)
        keypoints_data = results[0].keypoints

        # 1. KIỂM TRA SỰ HIỆN DIỆN CƠ BẢN
        valid_raw_keypoints_count = 0
        if results and keypoints_data and keypoints_data.shape[1] > 0:
            raw_keypoints_xy = keypoints_data.xy[0].cpu().numpy().astype(int)
            for idx in IMPORTANT_KEYPOINTS_INDICES:
                if idx < len(raw_keypoints_xy) and raw_keypoints_xy[idx, 0] > 1 and raw_keypoints_xy[idx, 1] > 1:
                    valid_raw_keypoints_count += 1
        
        if valid_raw_keypoints_count < MIN_RAW_KEYPOINTS_FOR_SITTING:
            return 2, keypoints_data # Status 2 (Absent)

        # 2. TRÍCH XUẤT ĐẶC TRƯNG
        features = extract_and_normalize_features(keypoints_data)
        
        if features is not None:
            if np.sum(features) == 0: 
                status_code = 2 # Vector rỗng -> Absent
            else:
                try:
                    features_reshaped = features.reshape(1, -1)
                    features_scaled = scaler.transform(features_reshaped)
                    
                    # 🐞 SỬA LỖI LOGIC AI (QUAN TRỌNG)
                    # Phải dự đoán trên 'features_scaled', không phải 'features_reshaped'
                    prediction = classifier.predict(features_scaled)[0] 
                    
                    status_code = int(prediction) # 0, 1, hoặc 2 (từ model 3 lớp)
                except Exception as e_svm:
                    status_code = 2 # Lỗi SVM -> Absent
        else:
            status_code = 2 # Không trích xuất/chuẩn hóa được -> Absent
            
    except Exception as e_yolo:
        status_code = 2
    
    return status_code, keypoints_data

# --- 6. LUỒNG XỬ LÝ ẢNH NỀN ---
def background_processor():
    global latest_frame_processed, posture_status, latest_keypoints
    global incorrect_posture_start_time, alert_sent_for_current_session
    global correct_posture_start_time
    global TOTAL_SITTING_TIME_SEC, TOTAL_MOVING_TIME_SEC

    while True:
        try:
            frame_to_process = frame_queue.get(block=True)
            current_status, current_keypoints = predict_posture(frame_to_process, yolo_model, scaler, classifier)
            
            posture_status = current_status
            with threading.Lock():
                latest_frame_processed = frame_to_process.copy()
                latest_keypoints = current_keypoints

            current_time = time.time()
            
            # Logic tính thời gian (Dựa trên kết quả model 3 lớp)
            if current_status == 0 or current_status == 1: # Ngồi (Đúng hoặc Sai)
                TOTAL_SITTING_TIME_SEC += TIME_PER_FRAME_SEC
            else: # Status 2 (Vắng mặt/Đứng/Lỗi)
                TOTAL_MOVING_TIME_SEC += TIME_PER_FRAME_SEC

            # Logic Cảnh báo/Reset (Giữ nguyên)
            if current_status == 1: # Sai
                correct_posture_start_time = None
                if incorrect_posture_start_time is None: incorrect_posture_start_time = current_time
                else:
                    duration_incorrect = current_time - incorrect_posture_start_time
                    if duration_incorrect >= INCORRECT_POSTURE_THRESHOLD_SECONDS and not alert_sent_for_current_session:
                        print(f"[{time.strftime('%H:%M:%S')}] !!! GỬI CẢNH BÁO TƯ THẾ SAI !!!")
                        alert_sent_for_current_session = True
            elif current_status == 0: # Đúng
                incorrect_posture_start_time = None
                if correct_posture_start_time is None: correct_posture_start_time = current_time
                else:
                    duration_correct = current_time - correct_posture_start_time
                    if alert_sent_for_current_session and duration_correct >= CORRECT_POSTURE_RESET_ALERT_SECONDS:
                        print(f"[{time.strftime('%H:%M:%S')}] Ngồi đúng đủ lâu, reset cờ cảnh báo.")
                        alert_sent_for_current_session = False
            else: # Không phát hiện / Đứng
                incorrect_posture_start_time = None
                correct_posture_start_time = None
                if alert_sent_for_current_session:
                    print(f"[{time.strftime('%H:%M:%S')}] Không thấy người/Đứng, reset cờ cảnh báo.")
                    alert_sent_for_current_session = False

            frame_queue.task_done()
        except Exception as e:
            print(f"Lỗi trong luồng xử lý nền: {e}")
            posture_status = 2
            time.sleep(1)

# --- 7. ROUTE NHẬN ẢNH TỪ ESP32/CLIENT ---
@app.route('/', methods=['POST'])
def receive_image():
    try:
        img_bytes = request.data
        if not img_bytes: return jsonify({"status": "error", "message": "No image data received"}), 400
        nparr = np.frombuffer(img_bytes, np.uint8)
        frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if frame is not None:
            # Code ESP32 đã lật ảnh (s->set_vflip(s, 1)), KHÔNG lật lại ở đây
            # frame = cv2.flip(frame, 0) 
            
            if frame.shape[1] != INPUT_WIDTH or frame.shape[0] != INPUT_HEIGHT:
                frame = cv2.resize(frame, TARGET_SIZE)
            try:
                try: frame_queue.get_nowait()
                except Empty: pass
                frame_queue.put_nowait(frame)
            except Full: pass

            # Trả về trạng thái và thời gian
            # (ESP32 sẽ nhận status_code 1 NẾU cảnh báo đang được kích hoạt)
            status_to_send_esp = 1 if alert_sent_for_current_session else posture_status
            message = "GOOD_POSTURE" if status_to_send_esp == 0 else "BAD_POSTURE_ALERT" if status_to_send_esp == 1 else "ABSENT/ERROR"

            return jsonify({
                "status_code": status_to_send_esp,
                "message": message,
                "sitting_time_min": TOTAL_SITTING_TIME_SEC / 60,
                "moving_time_min": TOTAL_MOVING_TIME_SEC / 60
            }), 200
        else:
            return jsonify({"status": "error", "message": "Lỗi giải mã JPEG"}), 400
    except Exception as e:
        print(f"Lỗi server khi nhận ảnh: {e}")
        return jsonify({"status": "error", "message": str(e)}), 500

# --- 8. HÀM HIỂN THỊ PREVIEW ---
def display_preview():
    global latest_frame_processed
    global TOTAL_SITTING_TIME_SEC, TOTAL_MOVING_TIME_SEC

    print("Khởi động cửa sổ xem trước (Nhấn 'q' để đóng)...")
    window_name = "Posture Detection Preview"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    while True:
        frame_to_show = None
        with threading.Lock():
            if latest_frame_processed is not None:
                frame_to_show = latest_frame_processed.copy()

        if frame_to_show is not None:
            # CHỈ VẼ TEXT (RẤT NHANH)
            status_text = "OK" if posture_status == 0 else "WRONG" if posture_status == 1 else "N/A"
            color = (0, 255, 0) if posture_status == 0 else (0, 0, 255) if posture_status == 1 else (255, 0, 0)
            cv2.putText(frame_to_show, f"Status: {status_text}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            sitting_min = TOTAL_SITTING_TIME_SEC / 60
            moving_min = TOTAL_MOVING_TIME_SEC / 60
            cv2.putText(frame_to_show, f"Sitting: {sitting_min:.1f} min", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            cv2.putText(frame_to_show, f"Moving: {moving_min:.1f} min", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            if alert_sent_for_current_session:
                cv2.putText(frame_to_show, "ALERT SENT!", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)

            cv2.imshow(window_name, frame_to_show)
        else:
            waiting_img = np.zeros((INPUT_HEIGHT, INPUT_WIDTH, 3), dtype=np.uint8)
            cv2.putText(waiting_img, "Waiting for image...", (10, INPUT_HEIGHT // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 1)
            cv2.imshow(window_name, waiting_img)

        if cv2.waitKey(10) & 0xFF == ord('q'): break

    cv2.destroyAllWindows()
    print("Đã đóng cửa sổ xem trước.")
    os._exit(0)

# --- 9. KHỞI ĐỘNG SERVER ---
if __name__ == '__main__':
    processor_thread = threading.Thread(target=background_processor, daemon=True)
    processor_thread.start()
    print("✅ Luồng xử lý nền đã khởi động.")

    if SHOW_PREVIEW_WINDOW:
        display_thread = threading.Thread(target=display_preview, daemon=True)
        display_thread.start()
        print("✅ Luồng hiển thị xem trước đã khởi động.")

    print(f"🔥 Server Flask đang chạy trên http://localhost:8000")
    print("   (Đảm bảo Cloudflare Tunnel của bạn đang trỏ đến localhost:8000)")
    print("   Nhấn Ctrl+C để dừng server.")
    try:
        from waitress import serve
        serve(app, host='0.0.0.0', port=8000, threads=8)
    except ImportError:
        print("\n--- Cảnh báo: waitress chưa được cài đặt, đang dùng server dev của Flask ---")
        app.run(host='0.0.0.0', port=8000, debug=False)


Đang tải mô hình...
❌ Lỗi tải model: [Errno 2] No such file or directory: 'posture_scaler_3class.joblib'
✅ Luồng xử lý nền đã khởi động.
Khởi động cửa sổ xem trước (Nhấn 'q' để đóng)...
✅ Luồng hiển thị xem trước đã khởi động.
🔥 Server Flask đang chạy trên http://localhost:8000
   (Đảm bảo Cloudflare Tunnel của bạn đang trỏ đến localhost:8000)
   Nhấn Ctrl+C để dừng server.

--- Cảnh báo: waitress chưa được cài đặt, đang dùng server dev của Flask ---
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://192.168.50.162:8000
Press CTRL+C to quit
127.0.0.1 - - [03/Nov/2025 17:09:49] "POST / HTTP/1.1" 200 -


Lỗi trong luồng xử lý nền: name 'scaler' is not defined


127.0.0.1 - - [03/Nov/2025 17:09:54] "POST / HTTP/1.1" 200 -


Lỗi trong luồng xử lý nền: name 'scaler' is not defined


...

127.0.0.1 - - [03/Nov/2025 17:10:24] "POST / HTTP/1.1" 200 -


Lỗi trong luồng xử lý nền: name 'scaler' is not defined


127.0.0.1 - - [03/Nov/2025 17:10:29] "POST / HTTP/1.1" 200 -


Lỗi trong luồng xử lý nền: name 'scaler' is not defined


: 